# Analysis

Runtime: **CPU, High-RAM**. No GPU. Run this only after the judge notebooks have finished and
`PREREG.md` is committed.

This is the **only** notebook that reads `error_type`. It:
1. records the commit hash of `PREREG.md`;
2. joins truth to detector output;
3. applies the 285-item flagging budget to every detector (PREREG 4);
4. builds D4 by confident learning over the stored option probabilities;
5. runs the estimators on the Redux arm;
6. runs the 200+ injection replicates and reports empirical coverage;
7. checks each of the four GO conditions.

The primary interval is the **nonparametric** bootstrap (PREREG 5a), a change from v1; the parametric one
is reported alongside it as a secondary.

Nothing in it may be retuned after looking at results. Anything not named in `PREREG.md` goes in the
exploratory section.

In [ ]:
!nproc
!pip -q install cleanlab numpy scipy joblib pyarrow
import os
print("CPU cores:", os.cpu_count(),
      "-- the replicate cell is CPU-bound; use the runtime with the most cores.")
from google.colab import drive
drive.mount("/content/drive")
import json
DRIVE = "/content/drive/MyDrive/label_error_crc/kill_v2"
print(sorted(os.listdir(DRIVE)))

In [ ]:
%%writefile crc.py
"""Capture-recapture estimators for counting label errors in a benchmark.

Each detector is one capture occasion. An item is "captured" by a detector only
if the detector flags it AND the flag is verified to be a real error, so the
population being estimated is the number of real label errors, including those
no detector flagged.

Estimators implemented
----------------------
union               naive baseline: the number of distinct verified errors found
chapman             bias-corrected Lincoln-Petersen, for one pair of detectors
loglinear           Fienberg (1972) multiple-systems model, main effects plus
                    two-way interactions chosen by forward AIC   [PRIMARY]
chao_lower          Chao (1984) lower bound under heterogeneous detectability
jackknife_mh        Burnham & Overton (1978, 1979) Mh jackknife

References are listed in literature/README.md. Every one has been opened and
verified. The jackknife coefficients are not transcribed from the paper: they
are re-derived here by Richardson extrapolation and checked against Burnham &
Overton's published closed forms for orders 1-3 in test_crc.py.

Only numpy and scipy are required.
"""

from __future__ import annotations

import itertools
from dataclasses import dataclass, field
from math import comb, log, sqrt

import numpy as np
from scipy import stats

__all__ = [
    "CaptureData",
    "Estimate",
    "union",
    "chapman",
    "loglinear",
    "chao_lower",
    "jackknife_mh",
    "summary",
]


# --------------------------------------------------------------------------
# data
# --------------------------------------------------------------------------


@dataclass
class CaptureData:
    """Capture histories of the verified errors that at least one detector found.

    `histories` is an (n_found, n_detectors) 0/1 array. Rows are verified errors,
    columns are detectors. An all-zero row is meaningless here (an error nobody
    found is by definition not in the data) and is rejected.
    """

    histories: np.ndarray
    detector_names: tuple[str, ...] = ()

    def __post_init__(self):
        self.histories = np.asarray(self.histories, dtype=np.int8)
        if self.histories.ndim != 2:
            raise ValueError("histories must be 2-D (items x detectors)")
        if self.histories.size and not np.isin(self.histories, (0, 1)).all():
            raise ValueError("histories must contain only 0 and 1")
        if self.histories.size and (self.histories.sum(axis=1) == 0).any():
            raise ValueError("an all-zero capture history cannot be observed")
        if not self.detector_names:
            self.detector_names = tuple(f"D{i + 1}" for i in range(self.n_detectors))
        if len(self.detector_names) != self.n_detectors:
            raise ValueError("detector_names length does not match histories width")

    @property
    def n_detectors(self) -> int:
        return self.histories.shape[1]

    @property
    def n_found(self) -> int:
        """D: distinct verified errors found by at least one detector."""
        return int(self.histories.shape[0])

    @property
    def frequencies(self) -> np.ndarray:
        """f[j-1] = number of errors captured by exactly j detectors, j = 1..K."""
        times = self.histories.sum(axis=1)
        return np.array(
            [int((times == j).sum()) for j in range(1, self.n_detectors + 1)]
        )

    def cell_counts(self) -> tuple[np.ndarray, np.ndarray]:
        """All 2^K - 1 observable patterns and their counts (zeros included)."""
        k = self.n_detectors
        patterns = np.array(
            [p for p in itertools.product((0, 1), repeat=k) if any(p)], dtype=np.int8
        )
        if self.n_found == 0:
            return patterns, np.zeros(len(patterns), dtype=float)
        index = {tuple(p): i for i, p in enumerate(map(tuple, patterns))}
        counts = np.zeros(len(patterns), dtype=float)
        for row in self.histories:
            counts[index[tuple(row)]] += 1
        return patterns, counts

    def bootstrap(self, rng: np.random.Generator) -> "CaptureData":
        """Nonparametric bootstrap: resample the found errors with replacement."""
        idx = rng.integers(0, self.n_found, size=self.n_found)
        return CaptureData(self.histories[idx], self.detector_names)

    @classmethod
    def from_flags(
        cls,
        flags: np.ndarray,
        is_error: np.ndarray,
        detector_names: tuple[str, ...] = (),
    ) -> "CaptureData":
        """Build capture histories from raw detector flags and verified truth.

        `flags` is (n_items, n_detectors) 0/1, `is_error` is (n_items,) 0/1.
        A capture is a flag on an item that is truly an error (a verified flag);
        flags on non-errors are false positives and are not captures.
        """
        flags = np.asarray(flags, dtype=np.int8)
        is_error = np.asarray(is_error, dtype=bool)
        captured = flags & is_error[:, None]
        rows = captured[captured.sum(axis=1) > 0]
        return cls(rows, detector_names)


@dataclass
class Estimate:
    """One estimator's answer. `n_hat` is the estimated total, found and unfound."""

    name: str
    n_hat: float
    lo: float = float("nan")
    hi: float = float("nan")
    ok: bool = True
    note: str = ""
    extra: dict = field(default_factory=dict)

    def covers(self, truth: float) -> bool:
        return bool(self.lo <= truth <= self.hi)


# --------------------------------------------------------------------------
# baseline and pairwise
# --------------------------------------------------------------------------


def union(data: CaptureData) -> Estimate:
    """The naive baseline: count what was actually found."""
    d = data.n_found
    return Estimate("union", float(d), float(d), float(d), note="baseline, not an estimator")


def chapman(data: CaptureData, i: int, j: int, conf: float = 0.95) -> Estimate:
    """Chapman's bias-corrected Lincoln-Petersen for detectors i and j.

    Cited through Rivest & Yauck (arXiv 2307.11705), which we have opened;
    Chapman (1951) itself could not be located (see literature/README.md).
    """
    a = data.histories[:, i].astype(bool)
    b = data.histories[:, j].astype(bool)
    n1, n2, m = int(a.sum()), int(b.sum()), int((a & b).sum())
    name = f"chapman[{data.detector_names[i]},{data.detector_names[j]}]"
    if n1 == 0 or n2 == 0:
        return Estimate(name, float("nan"), ok=False, note="a detector captured nothing")
    n_hat = (n1 + 1) * (n2 + 1) / (m + 1) - 1
    var = (
        (n1 + 1) * (n2 + 1) * (n1 - m) * (n2 - m) / ((m + 1) ** 2 * (m + 2))
    )
    se = sqrt(max(var, 0.0))
    z = stats.norm.ppf(0.5 + conf / 2)
    lo, hi = n_hat - z * se, n_hat + z * se
    note = "m = 0: unstable" if m == 0 else ""
    # A population cannot be smaller than what was observed, so BOTH ends are
    # floored at the union count. Flooring only the lower end (the v1 bug) made
    # a below-union estimate print as lo > hi.
    lo_f, hi_f = max(lo, data.n_found), max(hi, data.n_found)
    below = n_hat < data.n_found
    if below:
        note = (note + "; " if note else "") + (
            f"point estimate {n_hat:.1f} is below the union count {data.n_found}, "
            "which happens when the pair is strongly positively dependent; "
            "the interval is reported floored at the union")
    return Estimate(
        name,
        float(n_hat),
        float(lo_f),
        float(hi_f),
        ok=(m > 0 and not below),
        note=note,
        extra={"n1": n1, "n2": n2, "m": m, "se": se, "raw_lo": float(lo), "raw_hi": float(hi)},
    )


# --------------------------------------------------------------------------
# Chao lower bound
# --------------------------------------------------------------------------


def chao_lower(data: CaptureData, conf: float = 0.95) -> Estimate:
    """Chao (1984) lower bound, bias-corrected form, with a log-normal CI.

    This is a floor, not a point estimate: it is the smallest total consistent
    with the observed amount of single-capture mass. Reported as a sanity check
    on everything else.
    """
    d = data.n_found
    f = data.frequencies
    f1 = float(f[0]) if len(f) > 0 else 0.0
    f2 = float(f[1]) if len(f) > 1 else 0.0
    if f1 == 0:
        return Estimate("chao_lower", float(d), float(d), float(d), note="f1 = 0")
    # bias-corrected form, valid when f2 may be 0
    extra = f1 * (f1 - 1) / (2 * (f2 + 1))
    n_hat = d + extra
    # log-normal interval on the unseen part (Chao's standard construction)
    if extra <= 0:
        return Estimate("chao_lower", float(n_hat), float(d), float(n_hat), note="no unseen mass")
    var = (
        f1 * (f1 - 1) / (2 * (f2 + 1))
        + f1 * (2 * f1 - 1) ** 2 / (4 * (f2 + 1) ** 2)
        + f1**2 * f2 * (f1 - 1) ** 2 / (4 * (f2 + 1) ** 4)
    )
    se = sqrt(max(var, 1e-12))
    z = stats.norm.ppf(0.5 + conf / 2)
    c = np.exp(z * sqrt(log(1 + var / extra**2)))
    return Estimate(
        "chao_lower",
        float(n_hat),
        float(d + extra / c),
        float(d + extra * c),
        note="lower bound, not a point estimate",
        extra={"f1": f1, "f2": f2, "se": se},
    )


# --------------------------------------------------------------------------
# Burnham & Overton Mh jackknife
# --------------------------------------------------------------------------


def _jackknife_weights(k_occasions: int, order: int) -> np.ndarray:
    """Richardson-extrapolation weights that kill bias terms 1/K .. 1/K^order.

    Solves  sum_i w_i = 1  and  sum_i w_i (K-i)^-m = 0  for m = 1..order.
    test_crc.py checks that the resulting estimator equals Burnham & Overton's
    published closed forms for orders 1, 2 and 3.
    """
    n = order + 1
    a = np.empty((n, n), dtype=float)
    a[0, :] = 1.0
    for m in range(1, n):
        a[m, :] = [1.0 / (k_occasions - i) ** m for i in range(n)]
    b = np.zeros(n)
    b[0] = 1.0
    return np.linalg.solve(a, b)


def _jackknife_coefficients(k_occasions: int, order: int) -> np.ndarray:
    """a[j-1] in  N_hat_order = D + sum_j a[j-1] * f_j,  for j = 1..K.

    Derived from the delete-i jackknife, not transcribed: an item caught in
    exactly j occasions disappears when a random i-subset of occasions is
    deleted with probability C(K-j, i-j)/C(K, i), and the Richardson weights
    sum to 1, so the D terms cancel.
    """
    a = np.zeros(k_occasions)
    if order == 0:
        return a
    w = _jackknife_weights(k_occasions, order)
    for j in range(1, k_occasions + 1):
        a[j - 1] = -sum(
            w[i] * comb(k_occasions - j, i - j) / comb(k_occasions, i)
            for i in range(j, order + 1)
        )
    return a


def _jackknife_point(freqs: np.ndarray, d: int, k_occasions: int, order: int) -> float:
    """N-hat for the jackknife of the given order."""
    if order == 0:
        return float(d)
    return float(d + _jackknife_coefficients(k_occasions, order) @ np.asarray(freqs, float))


def _jackknife_select_order(freqs: np.ndarray, d: int, k: int, max_order: int, alpha: float) -> int:
    """Burnham & Overton's sequential test, with their variance of the difference.

    Start at order 0 (= take what was found) and step up while the next order
    differs significantly. Var(N_{o+1} - N_o) = sum_j (a_{o+1,j} - a_{o,j})^2 f_j
    under the usual multinomial approximation.
    """
    z_crit = stats.norm.ppf(1 - alpha / 2)
    f = np.asarray(freqs, dtype=float)
    for o in range(0, max_order):
        a_here = _jackknife_coefficients(k, o)
        a_next = _jackknife_coefficients(k, o + 1)
        diff = float((a_next - a_here) @ f)
        var = float(((a_next - a_here) ** 2) @ f)
        if var <= 0:
            return o
        if abs(diff) / sqrt(var) < z_crit:
            return o
    return max_order


def jackknife_mh(
    data: CaptureData,
    conf: float = 0.95,
    alpha: float = 0.05,
    n_boot: int = 2000,
    seed: int = 0,
    order: int | None = None,
) -> Estimate:
    """Burnham & Overton (1978, 1979) Mh jackknife.

    SECONDARY estimator. The order is chosen by Burnham & Overton's sequential
    test, and the interval is a nonparametric bootstrap in which the order is
    re-selected inside every replicate, so the interval pays for order-selection
    uncertainty as well as sampling noise.

    The simulation study in test_crc.py shows this estimator runs high and its
    interval is too narrow when detectability is close to homogeneous. It is
    reported for comparison; it is not the estimator the GO rule depends on.
    """
    d = data.n_found
    k = data.n_detectors
    f = data.frequencies
    max_order = min(k - 1, 5)
    if d == 0 or max_order < 1:
        return Estimate("jackknife_mh", float(d), ok=False, note="not enough detectors")

    chosen = int(order) if order is not None else _jackknife_select_order(
        f, d, k, max_order, alpha
    )
    n_hat = _jackknife_point(f, d, k, chosen)

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        rs = data.bootstrap(rng)
        fb, db = rs.frequencies, rs.n_found
        ob = int(order) if order is not None else _jackknife_select_order(
            fb, db, k, max_order, alpha
        )
        boots[b] = _jackknife_point(fb, db, k, ob)
    lo, hi = np.percentile(boots, [100 * (1 - conf) / 2, 100 * (1 + conf) / 2])

    clipped = n_hat < d
    return Estimate(
        "jackknife_mh",
        float(max(n_hat, d)),
        float(max(lo, d)),
        float(max(hi, d)),
        ok=not clipped,
        note=f"order {chosen}" + (" (clipped at D)" if clipped else ""),
        extra={"order": chosen, "frequencies": f.tolist()},
    )


# --------------------------------------------------------------------------
# log-linear multiple-systems estimation  (primary)
# --------------------------------------------------------------------------


def _poisson_irls(x: np.ndarray, y: np.ndarray, max_iter: int = 100, tol: float = 1e-10):
    """Poisson log-link GLM by IRLS. Returns (beta, converged)."""
    beta = np.zeros(x.shape[1])
    beta[0] = log(max(y.mean(), 1e-3))
    for _ in range(max_iter):
        eta = np.clip(x @ beta, -30, 30)
        mu = np.exp(eta)
        w = mu
        z = eta + (y - mu) / np.maximum(mu, 1e-10)
        xtw = x.T * w
        try:
            new = np.linalg.solve(xtw @ x + 1e-10 * np.eye(x.shape[1]), xtw @ z)
        except np.linalg.LinAlgError:
            return beta, False
        if not np.all(np.isfinite(new)):
            return beta, False
        if np.max(np.abs(new - beta)) < tol:
            return new, True
        beta = new
    return beta, False


def _design(patterns: np.ndarray, terms: list[tuple[int, ...]]) -> np.ndarray:
    cols = [np.ones(len(patterns))]
    for t in terms:
        col = np.ones(len(patterns))
        for idx in t:
            col = col * patterns[:, idx]
        cols.append(col.astype(float))
    return np.column_stack(cols)


def _fit(patterns, counts, terms):
    x = _design(patterns, terms)
    beta, ok = _poisson_irls(x, counts)
    mu = np.exp(np.clip(x @ beta, -30, 30))
    with np.errstate(divide="ignore", invalid="ignore"):
        dev_terms = np.where(counts > 0, counts * np.log(counts / mu), 0.0) - (counts - mu)
    deviance = float(2 * dev_terms.sum())
    aic = deviance + 2 * x.shape[1]
    return beta, ok, deviance, aic, x.shape[1]


def _forward_aic(patterns, counts, k, max_order=2):
    """Main effects, then greedily add the two-way term that lowers AIC most."""
    main = [(i,) for i in range(k)]
    candidates = [c for c in itertools.combinations(range(k), 2)] if max_order >= 2 else []
    terms = list(main)
    _, ok, _, best_aic, _ = _fit(patterns, counts, terms)
    improved = True
    while improved and candidates:
        improved = False
        best_term, best_new = None, best_aic
        for c in candidates:
            _, ok_c, _, aic_c, _ = _fit(patterns, counts, terms + [c])
            if ok_c and aic_c < best_new - 1e-9:
                best_term, best_new = c, aic_c
        if best_term is not None:
            terms.append(best_term)
            candidates.remove(best_term)
            best_aic = best_new
            improved = True
    return terms


def _all_patterns(k: int) -> np.ndarray:
    """All 2^k patterns, the all-zero (never-detected) cell first."""
    return np.array(list(itertools.product((0, 1), repeat=k)), dtype=np.int8)


def _parametric_draw(data: CaptureData, terms, rng: np.random.Generator) -> CaptureData | None:
    """Simulate a whole population from the fitted model, then keep what is seen.

    Unlike resampling the found errors, this regenerates the unobserved cell
    too, so the interval carries the uncertainty in the part of the population
    nobody detected. That is the part the estimate is actually about.
    """
    k = data.n_detectors
    patterns, counts = data.cell_counts()
    beta, ok, _, _, _ = _fit(patterns, counts, terms)
    if not ok:
        return None
    full = _all_patterns(k)
    mu = np.exp(np.clip(_design(full, terms) @ beta, -30, 30))
    total = mu.sum()
    if not np.isfinite(total) or total <= 0:
        return None
    n_star = rng.poisson(total)
    if n_star < 5:
        return None
    cell_counts = rng.multinomial(n_star, mu / total)
    rows = []
    for pattern, c in zip(full, cell_counts):
        if c and pattern.any():
            rows.append(np.repeat(pattern[None, :], c, axis=0))
    if not rows:
        return None
    return CaptureData(np.vstack(rows), data.detector_names)


def _loglinear_point(data: CaptureData, max_order: int, max_ratio: float):
    patterns, counts = data.cell_counts()
    k = data.n_detectors
    d = data.n_found
    terms = _forward_aic(patterns, counts, k, max_order)
    beta, ok, deviance, aic, n_par = _fit(patterns, counts, terms)
    unseen = float(np.exp(np.clip(beta[0], -30, 30)))
    n_hat = d + unseen
    stable = ok and np.isfinite(n_hat) and n_hat <= max_ratio * max(d, 1)
    if not stable:
        terms = [(i,) for i in range(k)]
        beta, ok, deviance, aic, n_par = _fit(patterns, counts, terms)
        unseen = float(np.exp(np.clip(beta[0], -30, 30)))
        n_hat = d + unseen
        n_hat = min(n_hat, max_ratio * max(d, 1))
    df = len(patterns) - n_par
    return n_hat, terms, deviance, df, stable


def loglinear(
    data: CaptureData,
    conf: float = 0.95,
    max_order: int = 2,
    n_boot: int = 2000,
    seed: int = 0,
    max_ratio: float = 10.0,
    ci_method: str = "nonparametric",
) -> Estimate:
    """PRIMARY estimator. Fienberg (1972) log-linear multiple-systems model.

    Main effects plus two-way detector interactions selected by forward AIC.
    Model selection is re-run inside every bootstrap replicate, so the interval
    pays for selection uncertainty as well as sampling noise.

    ci_method:
      "nonparametric" resample the found errors with replacement. PRE-REGISTERED
                      DEFAULT FOR v2. With four detectors and a D4 that is built
                      from D1-D3's own probabilities, the parametric interval
                      undercovers badly (74% at d4-dep, 84% at the
                      injection-sized d4-dep-n266), while this one holds 88-95%
                      in every scenario without an invisible subpopulation.
      "parametric"    simulate a whole population from the fitted model and keep
                      what is detected. This was v1's default, where five
                      detectors with an independent D5 made it the better of the
                      two. Reported as a secondary in v2.

    `max_ratio` is the stability guard: if the selected model implies more than
    `max_ratio` x (found errors), the fit is treated as unstable and the
    independence model is used instead. Fixed here, in advance, from simulation
    only -- never retuned after seeing expert labels.
    """
    d = data.n_found
    if d == 0:
        return Estimate("loglinear", 0.0, 0.0, 0.0, ok=False, note="nothing found")

    n_hat, terms, deviance, df, stable = _loglinear_point(data, max_order, max_ratio)
    gof_p = float(stats.chi2.sf(deviance, df)) if df > 0 else float("nan")

    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        if ci_method == "parametric":
            rs = _parametric_draw(data, terms, rng)
        elif ci_method == "nonparametric":
            rs = data.bootstrap(rng)
        else:
            raise ValueError(f"unknown ci_method {ci_method!r}")
        if rs is None:
            continue
        boots.append(_loglinear_point(rs, max_order, max_ratio)[0])
    boots = np.asarray(boots, dtype=float)
    if boots.size < 20:
        lo = hi = float("nan")
    else:
        lo, hi = np.percentile(boots, [100 * (1 - conf) / 2, 100 * (1 + conf) / 2])

    pretty = [":".join(data.detector_names[i] for i in t) for t in terms if len(t) > 1]
    return Estimate(
        "loglinear",
        float(n_hat),
        float(max(lo, d)) if np.isfinite(lo) else float("nan"),
        float(max(hi, d)) if np.isfinite(hi) else float("nan"),
        ok=stable,
        note=("interactions: " + ", ".join(pretty) if pretty else "independence model")
        + ("" if stable else "; UNSTABLE, fell back to independence"),
        extra={
            "terms": terms,
            "deviance": deviance,
            "df": df,
            "gof_p": gof_p,
            "ci_method": ci_method,
            "boot_sd": float(boots.std(ddof=1)) if boots.size > 1 else float("nan"),
            "n_boot_used": int(boots.size),
        },
    )


# --------------------------------------------------------------------------


def summary(
    data: CaptureData,
    conf: float = 0.95,
    n_boot: int = 2000,
    seed: int = 0,
    include_pairs: bool = True,
) -> list[Estimate]:
    """Run every estimator. The first entry is the pre-registered primary one."""
    out = [
        loglinear(data, conf=conf, n_boot=n_boot, seed=seed),
        jackknife_mh(data, conf=conf, n_boot=n_boot, seed=seed),
        chao_lower(data, conf=conf),
        union(data),
    ]
    if include_pairs:
        for i, j in itertools.combinations(range(data.n_detectors), 2):
            out.append(chapman(data, i, j, conf=conf))
    return out

In [ ]:
%%writefile detectors.py
"""Prompt construction, option shuffling, work-list building and verdict parsing.

This module holds everything a judge notebook needs that is not vLLM itself, so
that the fiddly parts -- the per-item option permutation, the choice of which 2
of 3 wrong keys to score, and verdict parsing -- can be tested on a laptop
without a GPU. `test_detectors.py` does exactly that.

Nothing here reads `error_type`. The work list a judge runs is built from the
questions, the options and the keys only, so a judge notebook never touches the
expert labels. Truth is joined to detector output in the analysis notebook,
after PREREG.md is committed.

See PREREG.md sections 3, 7. v2 differs from v1 in one decisive way: a detector
does not flag whatever it calls INCORRECT, it flags a FIXED BUDGET of its most
suspicious items, ranked by a score. v1 failed because the judges flagged 73% of
the benchmark; a budget makes a detector behave like a real audit.
"""

from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict

import numpy as np

MASTER_SEED = 20261006
LETTERS = ("A", "B", "C", "D")

# row kinds in the work list
REDUX = "redux"          # the item's real key, shuffled order      (5,700 per judge)
INJECT = "inject"        # an `ok` item with a wrong key, shuffled  (10,660 per judge)
ORIGINAL = "original"    # the real key, ORIGINAL order, 600 items  (section 9a)


# --------------------------------------------------------------------------
# per-item randomisation, fixed by seed
# --------------------------------------------------------------------------


def option_permutation(item_index: int) -> np.ndarray:
    """The fixed permutation of the 4 options for this item.

    Drawn from (MASTER_SEED, item_index) so it is identical for every judge,
    for the confident-learning pass, for injection scoring and for every
    replicate. `perm[new_position] = old_position`.
    """
    rng = np.random.default_rng([MASTER_SEED, int(item_index)])
    return rng.permutation(4)


def scored_wrong_keys(item_index: int, answer: int) -> list[int]:
    """The 2 of the 3 wrong options this item is scored against (PREREG 7).

    Drawn at random per item rather than by position, so no option position is
    favoured. Returned as ORIGINAL option indices, sorted for stability.
    """
    wrong = [i for i in range(4) if i != int(answer)]
    rng = np.random.default_rng([MASTER_SEED, int(item_index), 1])
    picked = rng.choice(len(wrong), size=2, replace=False)
    return sorted(wrong[i] for i in picked)


def apply_permutation(options: list[str], key_index: int, perm: np.ndarray):
    """Reorder the options and move the key with its text.

    Returns (reordered_options, new_key_index). The caller labels the reordered
    options A-D in order, so the claimed key is named by its new letter and the
    original A-D positions carry no information.
    """
    reordered = [options[i] for i in perm]
    new_key = int(np.where(perm == int(key_index))[0][0])
    return reordered, new_key


# --------------------------------------------------------------------------
# the work list
# --------------------------------------------------------------------------


@dataclass
class Row:
    """One prompt a judge must answer."""

    row_id: str
    kind: str
    item_index: int
    subject: str
    claimed_key_original: int   # which ORIGINAL option is being claimed correct
    shuffled: bool

    def to_json(self) -> str:
        return json.dumps(asdict(self), sort_keys=True)


def build_work_list(items: list[dict], original_order_sample: set[int]) -> list[Row]:
    """Every prompt one judge runs, in a fixed order.

    `items` must be the pinned dataset in a fixed order, each dict carrying
    `subject`, `answer` and `choices`. `original_order_sample` holds the item
    indices in the 600-item section 9a sample.

    Per judge this is 5,700 redux rows + 2 x (number of `ok` items) inject rows
    + 600 original-order rows. The `ok` set is passed in by the caller as
    `item["is_ok"]`; see the note in build_inject_rows about why that is the one
    label the work list depends on.
    """
    rows: list[Row] = []
    for idx, item in enumerate(items):
        rows.append(
            Row(f"redux:{idx}", REDUX, idx, item["subject"], int(item["answer"]), True)
        )
    rows.extend(build_inject_rows(items))
    for idx in sorted(original_order_sample):
        item = items[idx]
        rows.append(
            Row(f"original:{idx}", ORIGINAL, idx, item["subject"], int(item["answer"]), False)
        )
    return rows


def build_inject_rows(items: list[dict]) -> list[Row]:
    """The injection-scoring rows: each `ok` item against 2 of its 3 wrong keys.

    This is the one place the work list depends on a label, because only `ok`
    items can carry an injected error -- flipping the key of an item that is
    already wrong would not give a known truth. It uses `is_ok` and nothing
    else: no error type, no correct_answer, no expert flag. The analysis
    notebook is still the only place truth is joined to detector output.
    """
    rows: list[Row] = []
    for idx, item in enumerate(items):
        if not item.get("is_ok"):
            continue
        for wrong in scored_wrong_keys(idx, int(item["answer"])):
            rows.append(
                Row(f"inject:{idx}:{wrong}", INJECT, idx, item["subject"], wrong, True)
            )
    return rows


def original_order_sample(items: list[dict], key_error_indices: list[int], n_total: int = 600) -> list[int]:
    """The section 9a sample: all key errors plus a stratified top-up.

    Built in the analysis notebook (it needs the expert labels to know which
    items are key errors) and handed to the judge notebooks as a plain list of
    indices, so a judge still never sees a label.
    """
    chosen = set(int(i) for i in key_error_indices)
    remaining = n_total - len(chosen)
    if remaining <= 0:
        return sorted(chosen)
    pool: dict[str, list[int]] = {}
    for idx, item in enumerate(items):
        if idx in chosen:
            continue
        pool.setdefault(item["supercategory"], []).append(idx)
    sizes = {k: len(v) for k, v in pool.items()}
    total = sum(sizes.values())
    if total == 0:
        return sorted(chosen)
    remaining = min(remaining, total)
    # proportional allocation with largest-remainder rounding, so the top-up is
    # stratified by supercategory and the total is exact
    exact = {k: remaining * n / total for k, n in sizes.items()}
    take = {k: int(v) for k, v in exact.items()}
    for k in sorted(exact, key=lambda k: (-(exact[k] - take[k]), k)):
        if sum(take.values()) >= remaining:
            break
        if take[k] < sizes[k]:
            take[k] += 1
    rng = np.random.default_rng([MASTER_SEED, 9])
    for k in sorted(take):
        n = min(take[k], sizes[k])
        if n:
            picked = rng.choice(len(pool[k]), size=n, replace=False)
            chosen.update(pool[k][i] for i in picked)
    return sorted(chosen)


# --------------------------------------------------------------------------
# prompts
# --------------------------------------------------------------------------


JUDGE_INSTRUCTION = (
    "You are checking a multiple-choice exam question for errors in its answer key.\n"
    "Decide whether the claimed answer is the correct answer to the question.\n"
    "Reply with exactly one word after VERDICT: either CORRECT or INCORRECT.\n"
    "Do not explain. Do not think out loud. The first word you write is the verdict."
)

# The assistant turn is prefilled with this, so the FIRST generated token is the
# verdict word itself and its logprobs give P(INCORRECT) directly (PREREG 4).
# Without the prefill the first token would be "VERD", which carries no signal.
VERDICT_PREFILL = "VERDICT:"


def judge_prompt(question: str, options: list[str], claimed_letter: str) -> str:
    """The judge prompt. Identical template for every model (PREREG 3)."""
    lines = [JUDGE_INSTRUCTION, "", f"Question: {question.strip()}", ""]
    for letter, text in zip(LETTERS, options):
        lines.append(f"{letter}. {str(text).strip()}")
    lines += ["", f"Claimed answer: {claimed_letter}", ""]
    return "\n".join(lines)


def first_token_is_verdict(token_logprobs: dict[str, float]) -> bool:
    """True when a verdict word is among the first token's candidates.

    The notebook runs this on 8 items before the real pass. v1 lost its Qwen3.8
    arm to a thinking model that never reached a verdict; this catches that in
    seconds instead of after a full pass.
    """
    return incorrect_probability(token_logprobs) is not None


def answer_prompt(question: str, options: list[str]) -> str:
    """The confident-learning prompt: answer the question, no key shown."""
    lines = ["Answer the multiple-choice question. Reply with one letter only.", "",
             f"Question: {question.strip()}", ""]
    for letter, text in zip(LETTERS, options):
        lines.append(f"{letter}. {str(text).strip()}")
    lines += ["", "Answer:"]
    return "\n".join(lines)


def render_row(row: Row, item: dict) -> tuple[str, dict]:
    """Build the prompt for one work row, plus the metadata needed to read it back.

    Returns (prompt_text, meta). `meta["perm"]` records the permutation used, so
    the analysis notebook can map letters back to original option identities.
    """
    options = list(item["choices"])
    claimed = int(row.claimed_key_original)
    if row.shuffled:
        perm = option_permutation(row.item_index)
        options, claimed_new = apply_permutation(options, claimed, perm)
    else:
        perm = np.arange(4)
        claimed_new = claimed
    prompt = judge_prompt(item["question"], options, LETTERS[claimed_new])
    return prompt, {"perm": [int(x) for x in perm], "claimed_letter": LETTERS[claimed_new]}


# --------------------------------------------------------------------------
# parsing
# --------------------------------------------------------------------------


_VERDICT_RE = re.compile(r"VERDICT\s*[:\-]?\s*\**\s*(CORRECT|INCORRECT)", re.I)

CORRECT = "CORRECT"
INCORRECT = "INCORRECT"
UNPARSED = "unparsed"


def parse_verdict(text: str) -> str:
    """Read a judge's verdict from the `VERDICT:` line ONLY.

    v2 drops v1's bare-word fallback. In v1 that fallback could read "correct"
    out of a sentence of reasoning and turn it into a verdict; it fired on only
    3 of 67,840 answers so it changed nothing there, but with scores driving the
    ranking in v2 there is no reason to keep a lossy path. Anything without a
    `VERDICT:` line is `unparsed`.

    The verdict is the SECONDARY, reported-only signal in v2 (PREREG 3). The
    primary flag rule uses `incorrect_probability`, not this.
    """
    if not text:
        return UNPARSED
    m = _VERDICT_RE.search(text)
    if m:
        return m.group(1).upper()
    return UNPARSED


def is_flag(verdict: str) -> bool:
    """A detector flags the item when it says the claimed key is INCORRECT."""
    return verdict == INCORRECT


# --------------------------------------------------------------------------
# the automatic smoke check (PREREG 10, 12)
# --------------------------------------------------------------------------

MAX_UNPARSED = 0.05  # PREREG 12: above this the model is reported as broken


@dataclass
class SmokeResult:
    """The verdict of the after-first-chunk check. `ok` False means stop."""

    ok: bool
    unparsed_rate: float
    prompts_per_s: float
    projected_hours: float
    budget_hours: float
    reasons: list[str]
    message: str


def smoke_check(
    verdicts: list[str],
    session_elapsed_s: float,
    generate_elapsed_s: float,
    n_done: int,
    n_total: int,
    budget_hours: float,
    label: str = "",
    max_unparsed: float = MAX_UNPARSED,
) -> SmokeResult:
    """Decide, after the first chunk, whether this notebook may keep going.

    Two pre-registered stop conditions, both fixed in advance and neither
    depending on any label:

    * unparsed rate above 5% -- PREREG 12 says the model is then reported as
      broken, not quietly treated as a detector that flags nothing;
    * projected session hours past this notebook's share of the GPU budget --
      PREREG 10.

    The projection is in *session* hours, not GPU-busy hours, because Colab
    bills session time: it is the time already spent plus the time the
    remaining rows will take at the measured rate.
    """
    n = max(len(verdicts), 1)
    unparsed_rate = sum(v == UNPARSED for v in verdicts) / n
    rate = n_done / generate_elapsed_s if generate_elapsed_s > 0 else 0.0
    remaining = max(n_total - n_done, 0)
    projected_hours = (
        (session_elapsed_s + remaining / rate) / 3600.0 if rate > 0 else float("inf")
    )

    reasons = []
    if unparsed_rate > max_unparsed:
        reasons.append(
            f"unparsed rate {unparsed_rate:.1%} is above the {max_unparsed:.0%} limit "
            f"(PREREG 12). The prompt or the parser does not fit this model. "
            f"Do not treat it as a detector that flags nothing."
        )
    if projected_hours > budget_hours:
        reasons.append(
            f"projected {projected_hours:.2f} session hours is past this notebook's "
            f"budget of {budget_hours:.2f} (PREREG 10) at {rate:.1f} prompts/s."
        )

    head = f"SMOKE CHECK {label}".strip()
    body = (
        f"  rows done        {n_done} of {n_total}\n"
        f"  unparsed rate    {unparsed_rate:.1%}  (limit {max_unparsed:.0%})\n"
        f"  prompts/s        {rate:.1f}\n"
        f"  projected hours  {projected_hours:.2f}  (budget {budget_hours:.2f})"
    )
    if reasons:
        message = (
            f"{head}: STOP\n{body}\n\n"
            + "\n".join(f"  * {r}" for r in reasons)
            + "\n\n  Nothing is lost: what is on Drive is kept and the notebook resumes.\n"
            "  Report this before changing anything. Do not retune the prompt or the\n"
            "  parser to make the number go away without saying so."
        )
    else:
        message = f"{head}: PASS, continuing automatically\n{body}"
    return SmokeResult(
        ok=not reasons,
        unparsed_rate=unparsed_rate,
        prompts_per_s=rate,
        projected_hours=projected_hours,
        budget_hours=budget_hours,
        reasons=reasons,
        message=message,
    )


# --------------------------------------------------------------------------
# v2: scores, not verdicts (PREREG 3)
# --------------------------------------------------------------------------

FLAG_BUDGET = 285          # top 5% of 5,700 items, per detector
VERDICT_TOKENS = ("CORRECT", "INCORRECT")


def _logsumexp(values: list[float]) -> float:
    if not values:
        return float("-inf")
    m = max(values)
    if m == float("-inf"):
        return m
    return m + float(np.log(sum(np.exp(np.array(values) - m))))


def incorrect_probability(token_logprobs: dict[str, float]) -> float | None:
    """P(INCORRECT) from the logprobs of the first generated verdict token.

    `token_logprobs` maps a decoded token string to its logprob at the position
    where the verdict word starts. We renormalise over just the two verdict
    words, so the score is "given that it said one of the two, how sure was it
    that the key is wrong". Case and leading whitespace are ignored, and the
    `INCORRECT`/`CORRECT` substring trap is handled by exact matching on the
    stripped, upper-cased token.

    Returns None when neither verdict token appears in the top-k logprobs; the
    caller records that item as unscored. Unscored items sort last and so are
    never flagged, which is the conservative direction.
    """
    buckets: dict[str, list[float]] = {t: [] for t in VERDICT_TOKENS}
    for token, logprob in token_logprobs.items():
        key = str(token).strip().upper()
        if key in buckets:
            buckets[key].append(float(logprob))
    totals = {t: _logsumexp(v) for t, v in buckets.items() if v}
    if "INCORRECT" not in totals and "CORRECT" not in totals:
        return None
    li = totals.get("INCORRECT", float("-inf"))
    lc = totals.get("CORRECT", float("-inf"))
    denom = _logsumexp([li, lc])
    return float(np.exp(li - denom))


def top_k_flags(scores: dict[int, float | None], k: int = FLAG_BUDGET) -> set[int]:
    """Flag exactly the k highest-scoring items. Deterministic.

    Ties are broken by item index, ascending, so the same scores always give the
    same flag set regardless of dict ordering. Items with no score (None) sort
    last and are only reached if there are fewer than k scored items.
    """
    ordered = sorted(
        scores.items(),
        key=lambda kv: (kv[1] is None, -(kv[1] if kv[1] is not None else 0.0), kv[0]),
    )
    return {idx for idx, _ in ordered[:k]}


def self_confidence(probs: "np.ndarray", claimed_key: int) -> float:
    """D4's ranking score: how confident the ensemble is in the claimed key.

    Low means suspicious, so D4 flags the k LOWEST values. `probs` is the
    averaged option distribution over the four original option positions.
    """
    return float(np.asarray(probs, dtype=float)[int(claimed_key)])


def bottom_k_flags(confidence: dict[int, float], k: int = FLAG_BUDGET) -> set[int]:
    """Flag the k items with the lowest confidence in the claimed key.

    Ties break by item index ascending, as in `top_k_flags`.
    """
    ordered = sorted(confidence.items(), key=lambda kv: (kv[1], kv[0]))
    return {idx for idx, _ in ordered[:k]}


# --------------------------------------------------------------------------
# thinking must be OFF: v1 lost Qwen3.8 to it (RESULTS.md, section 9b)
# --------------------------------------------------------------------------

# Extra kwargs passed to tokenizer.apply_chat_template per model family.
# v1 lost the Qwen3.8 arm because the model spent all 24 output tokens
# reasoning and never reached a verdict. Every v2 judge must emit the verdict
# as its FIRST generated token; `test_first_token_is_verdict` in the notebook
# checks that on 8 items before the real pass starts.
THINKING_OFF = {
    # Qwen3.x: the template prefills an empty <think></think> block.
    "qwen": {"enable_thinking": False},
    # Mistral Small 4 is hybrid; its template accepts only 'none' or 'high'
    # and already defaults to 'none'. Passed explicitly so it cannot drift.
    "mistral": {"reasoning_effort": "none"},
    # Gemma 4 has no thinking switch in its template.
    "gemma": {},
}


def thinking_kwargs(model_id: str) -> dict:
    low = model_id.lower()
    for key, kwargs in THINKING_OFF.items():
        if key in low:
            return dict(kwargs)
    return {}

## Data, with labels this time

In [ ]:
# The pinned dataset. Downloads the 57 Arrow files at the exact revision in DATA.md.
# `is_ok` is the ONLY label used here, and only to decide which items can carry an
# injected error (PREREG 7). No error type, no correct_answer, no expert flag.
import json, urllib.parse, urllib.request, os
import pyarrow as pa, pyarrow.ipc as ipc

REVISION = "372ea425445d51e1ba1188c56e5e893f8138621f"
DATASET = "edinburgh-dawg/mmlu-redux-2.0"
CACHE = "/content/mmlu_redux"
os.makedirs(CACHE, exist_ok=True)

tree = json.load(urllib.request.urlopen(
    f"https://huggingface.co/api/datasets/{DATASET}/tree/{REVISION}?recursive=true", timeout=60))
paths = sorted(e["path"] for e in tree if e["type"] == "file" and e["path"].endswith(".arrow"))
assert len(paths) == 57, f"expected 57 arrow files, got {len(paths)}"

# supercategories from the official MMLU map, same source as DATA.md
src = urllib.request.urlopen(
    "https://raw.githubusercontent.com/hendrycks/test/master/categories.py", timeout=60).read().decode()
ns = {}
exec(compile(src, "categories.py", "exec"), ns)
SUB2CAT = {s: c for s, subs in ns["subcategories"].items()
            for c, members in ns["categories"].items() if subs[0] in members}

ITEMS_FULL = []
for path in paths:
    subject = path.split("/")[0]
    local = os.path.join(CACHE, subject + ".arrow")
    if not os.path.exists(local):
        urllib.request.urlretrieve(
            f"https://huggingface.co/datasets/{DATASET}/resolve/{REVISION}/" + urllib.parse.quote(path),
            local)
    with pa.memory_map(local, "rb") as fh:
        cols = ipc.open_stream(fh).read_all().to_pydict()
    for i in range(len(cols["question"])):
        ITEMS_FULL.append({
            "subject": subject,
            "supercategory": SUB2CAT[subject],
            "question": cols["question"][i],
            "choices": list(cols["choices"][i]),
            "answer": int(cols["answer"][i]),
            "is_ok": cols["error_type"][i] == "ok",
            "error_type": cols["error_type"][i],
        })

ITEMS = ITEMS_FULL
assert len(ITEMS) == 5700, len(ITEMS)
print(len(ITEMS), "items;", sum(i["is_ok"] for i in ITEMS), "ok items")

## Join truth

In [ ]:
# Join truth to detector output. THIS IS THE ONLY PLACE EXPERT LABELS ARE READ.
# Record the commit hash of PREREG.md first (PREREG 9, 13).
# Blob hash of experiments/kill_v2/PREREG.md as committed. Filled in by
# make_notebook.py so the author never needs to run git in Colab (PREREG 9).
PREREG_COMMIT = "d983111828355c30e2ab7bb5884a6b371624ff53"
assert PREREG_COMMIT, "record the pre-registration blob hash before running the analysis"
print("pre-registration blob:", PREREG_COMMIT)

PRIMARY = {"wrong_groundtruth", "no_correct_answer", "multiple_correct_answers"}
EXPERT = "expert"

truth = []
for i, item in enumerate(ITEMS_FULL):
    et = item["error_type"]
    truth.append({
        "item_index": i,
        "subject": item["subject"],
        "supercategory": item["supercategory"],
        "error_type": et,
        "is_key_error": et in PRIMARY,
        "is_expert": et == EXPERT,
        "is_ok": et == "ok",
    })
n_key = sum(t["is_key_error"] for t in truth)
n_expert = sum(t["is_expert"] for t in truth)
print(f"{n_key} key errors, {n_expert} expert items excluded from primary truth")
assert n_key == 181, f"expected 181 key errors at the pinned revision, got {n_key}"

## Load detector outputs and build D5

In [ ]:
import detectors as D
# Load the three judges' SCORES and their option probabilities (PREREG 3, 4).
# Labels are NOT used in this cell.
import numpy as np, collections
JUDGE_SHORTS = ["gemma4", "qwen38", "mistral4"]                 # D1..D3
DETECTOR_NAMES = ("D1_gemma4", "D2_qwen38", "D3_mistral4", "D4_cleanlab")
N_DET = 4
BUDGET = D.FLAG_BUDGET                                           # 285, PREREG 4
N = len(ITEMS)
ANS = np.array([int(it["answer"]) for it in ITEMS])

def _read_jsonl(path):
    out = {}
    with open(path) as fh:
        for line in fh:
            try:
                r = json.loads(line)
            except Exception:
                continue            # a half-written last line after a disconnect
            out[r.get("row_id", r.get("item_index"))] = r   # resume can repeat: last wins
    return out

REDUX_SCORE = [dict() for _ in JUDGE_SHORTS]   # item -> P(INCORRECT) on its real key
INJ_SCORE = [dict() for _ in JUDGE_SHORTS]     # (item, wrong key) -> P(INCORRECT)
REDUX_VERDICT = {}                             # secondary, reported only
UNSCORED = {}
for j, m in enumerate(JUDGE_SHORTS):
    rows = _read_jsonl(os.path.join(DRIVE, f"judge_{m}.jsonl")).values()
    kinds = collections.Counter(r["kind"] for r in rows)
    redux = {r["item_index"]: r for r in rows if r["kind"] == "redux"}
    assert len(redux) == N, f"{m}: {len(redux)} redux rows, expected {N} -- judge not finished"
    REDUX_SCORE[j] = {i: r["score"] for i, r in redux.items()}
    REDUX_VERDICT[m] = {i: r["verdict"] for i, r in redux.items()}
    for r in rows:
        if r["kind"] == "inject":
            INJ_SCORE[j][(r["item_index"], r["claimed_key_original"])] = r["score"]
    n_uns = sum(v is None for v in REDUX_SCORE[j].values())
    UNSCORED[m] = n_uns / N
    print(f"{m}: rows {dict(kinds)}  unscored on Redux {UNSCORED[m]:.2%}  "
          f"inject scores {len(INJ_SCORE[j])}")
    if UNSCORED[m] > 0.05:
        print(f"  WARNING: {m} unscored rate above 5% -- reported as broken (PREREG 4)")

# D4's source: average the three models' option probabilities (already in original
# option order). PREREG 3.
P = np.zeros((N, 4))
for m in JUDGE_SHORTS:
    pr = _read_jsonl(os.path.join(DRIVE, f"probs_{m}.jsonl"))
    assert len(pr) == N, f"{m}: probability pass has {len(pr)} of {N} items"
    for i, r in pr.items():
        P[int(i)] += np.asarray(r["probs"], dtype=float)
P /= len(JUDGE_SHORTS)
P = P / P.sum(axis=1, keepdims=True)

def build_flag_matrix(flipped_key):
    """(N, 4) flags for a benchmark whose keys are ANS except the flipped items.

    THE BUDGET IS APPLIED PER BENCHMARK (PREREG 4): each judge flags exactly its
    top 285 items by P(INCORRECT), and D4 the 285 items whose claimed key the
    ensemble is least confident in. Flipped items take their injection score,
    every other item its Redux score.
    """
    labels = ANS.copy()
    for i, k in flipped_key.items():
        labels[i] = k
    flags = np.zeros((N, N_DET), dtype=np.int8)
    for j in range(len(JUDGE_SHORTS)):
        scores = dict(REDUX_SCORE[j])
        for i, k in flipped_key.items():
            scores[i] = INJ_SCORE[j][(i, k)]
        for i in D.top_k_flags(scores, BUDGET):
            flags[i, j] = 1
    conf = {i: float(P[i, labels[i]]) for i in range(N)}
    for i in D.bottom_k_flags(conf, BUDGET):
        flags[i, 3] = 1
    assert flags.sum(axis=0).tolist() == [BUDGET] * N_DET, flags.sum(axis=0)
    return flags

print("loaded; inject scores per judge:", [len(d) for d in INJ_SCORE])

## Redux arm: the realism test

In [ ]:
# The Redux arm: realism test against the 181 expert-labelled key errors (PREREG 2, 4, 5, 8).
import crc
MASTER_SEED = 20261006
is_key = np.array([t["is_key_error"] for t in truth])
is_expert = np.array([t["is_expert"] for t in truth])
is_ok = np.array([t["is_ok"] for t in truth])
FLAGS_REDUX = build_flag_matrix({})

print("per detector on Redux (precision excludes the 32 expert items):")
for j, name in enumerate(DETECTOR_NAMES):
    f = FLAGS_REDUX[:, j].astype(bool)
    tp = int((f & is_key).sum()); fp = int((f & ~is_key & ~is_expert).sum())
    print(f"  {name:12s} flags {int(f.sum()):5d}  captures {tp:4d}  recall {tp/181:6.1%}  "
          f"precision {tp/max(tp+fp,1):6.1%}  flags on ok items {int((f & is_ok).sum()):5d}")

data_redux = crc.CaptureData.from_flags(FLAGS_REDUX, is_key, DETECTOR_NAMES)
pat, cnt = data_redux.cell_counts()
print("\ncapture-pattern table (D1..D4), 15 cells:")
for p_, c_ in zip(pat, cnt):
    print("  ", "".join(map(str, p_)), int(c_))

REDUX_EST = crc.summary(data_redux, n_boot=2000, seed=MASTER_SEED)
PRIMARY_REDUX = REDUX_EST[0]
print(f"\nunion found {data_redux.n_found} of 181 key errors")
for e in REDUX_EST:
    gof = e.extra.get("gof_p")
    print(f"  {e.name:22s} n_hat {e.n_hat:8.1f}  CI [{e.lo:7.1f}, {e.hi:7.1f}]  ok={e.ok}  "
          + (f"gof_p={gof:.3f}  " if gof is not None and gof == gof else "") + e.note)
print(f"\nestimated key errors NO detector found: {PRIMARY_REDUX.n_hat - data_redux.n_found:.1f} "
      f"[{PRIMARY_REDUX.lo - data_redux.n_found:.1f}, {PRIMARY_REDUX.hi - data_redux.n_found:.1f}]")
union_flags = int(FLAGS_REDUX.any(axis=1).sum())
print(f"audit cost: a human checks {union_flags} flagged items to find {data_redux.n_found} key errors")

print("\nsecondary, by supercategory (not part of GO):")
cats = np.array([t["supercategory"] for t in truth])
for c in sorted(set(cats)):
    m_ = cats == c
    d_ = crc.CaptureData.from_flags(FLAGS_REDUX[m_], is_key[m_], DETECTOR_NAMES)
    e_ = crc.loglinear(d_, n_boot=2000, seed=MASTER_SEED)
    print(f"  {c:34s} truth {int(is_key[m_].sum()):3d}  union {d_.n_found:3d}  "
          f"n_hat {e_.n_hat:7.1f}  CI [{e_.lo:6.1f}, {e_.hi:6.1f}]  covers {e_.covers(int(is_key[m_].sum()))}")

print("\nSECONDARY, reported only, never used for GO (PREREG 4):")
print("what the natural INCORRECT verdict would have flagged, had there been no budget:")
for j, m in enumerate(JUDGE_SHORTS):
    nat = np.array([REDUX_VERDICT[m].get(i) == "INCORRECT" for i in range(N)])
    tp = int((nat & is_key).sum()); fp = int((nat & ~is_key & ~is_expert).sum())
    print(f"  {m:9s} would flag {int(nat.sum()):5d}  captures {tp:4d}  recall {tp/181:6.1%}  "
          f"precision {tp/max(tp+fp,1):6.1%}")
nat_union = np.zeros(N, dtype=bool)
for m in JUDGE_SHORTS:
    nat_union |= np.array([REDUX_VERDICT[m].get(i) == "INCORRECT" for i in range(N)])
print(f"  verdict-union would flag {int(nat_union.sum())} items and catch "
      f"{int((nat_union & is_key).sum())} of 181 -- compare with the budgeted union above.")
print("  (v1 failed here: its verdict-union flagged 4,145 of 5,700 and caught 94.5%.)")

## Injection replicates: the main calibration test

In [ ]:
# The 200+ injection replicates (PREREG 7). CPU only; parallel over seeds.
import numpy as np
from joblib import Parallel, delayed
import crc
import detectors            # used inside one_replicate, which runs in a worker
import detectors as D       # build_flag_matrix closes over this name

MASTER_SEED = 20261006
N_REPLICATES = 200
N_BOOT = 2000          # PREREG 9; the local end-to-end test overrides both
PER_STRATUM = {"STEM": 86, "humanities": 60, "social sciences": 57,
               "other (business, health, misc.)": 63}  # 266 total, PREREG 7

ok_by_cat = {}
for t in truth:
    if t["is_ok"]:
        ok_by_cat.setdefault(t["supercategory"], []).append(t["item_index"])

def one_replicate(rep):
    rng = np.random.default_rng(MASTER_SEED + rep)
    injected = []
    for cat, n in PER_STRATUM.items():
        pool = ok_by_cat[cat]
        injected.extend(int(x) for x in rng.choice(pool, size=n, replace=False))
    injected = set(injected)
    # which wrong key each injected item carries, drawn from the 2 that were scored
    flipped_key = {}
    for i in sorted(injected):
        pair = detectors.scored_wrong_keys(i, ITEMS[i]["answer"])
        flipped_key[i] = int(rng.choice(pair))
    flags = build_flag_matrix(flipped_key)          # (5700, 4) 0/1, budget applied per replicate
    is_error = np.zeros(len(ITEMS), dtype=bool)
    is_error[sorted(injected)] = True
    data = crc.CaptureData.from_flags(flags, is_error, DETECTOR_NAMES)
    # PRIMARY: nonparametric bootstrap (PREREG 5a). crc.loglinear defaults to it.
    est = crc.loglinear(data, n_boot=N_BOOT, seed=MASTER_SEED)
    alt = crc.loglinear(data, n_boot=N_BOOT, seed=MASTER_SEED, ci_method="parametric")
    return {
        "rep": rep, "truth": len(injected), "union": data.n_found,
        "n_hat": est.n_hat, "lo": est.lo, "hi": est.hi, "ok": est.ok,
        "gof_p": est.extra["gof_p"], "terms": est.extra["terms"],
        "covers": est.covers(len(injected)),
        "covers_parametric": alt.covers(len(injected)),   # secondary, reported only
    }

import os as _os
print("CPU cores available:", _os.cpu_count(),
      "-- this cell is CPU-bound; pick the runtime with the most cores.")
results = Parallel(n_jobs=-1, verbose=5)(
    delayed(one_replicate)(r) for r in range(N_REPLICATES))
coverage = np.mean([r["covers"] for r in results])
cov_par = np.mean([r["covers_parametric"] for r in results])
print(f"empirical coverage over {N_REPLICATES} replicates: {coverage:.1%}  (GO needs >= 85%)")
print(f"  secondary, parametric interval: {cov_par:.1%}  (reported only, PREREG 5a)")

## GO / NO-GO

Check all four conditions from `PREREG.md` section 8 and write `RESULTS.md`:

- (a) coverage over the replicates >= 85%
- (b) the pooled Redux interval covers 181
- (c) the union misses >= 20% of the 181
- (d) |estimate - truth| <= 0.5 |union - truth|, on Redux and averaged over replicates

Report every detector's precision and recall, the full 31-cell capture pattern table, every estimator
with its goodness of fit, the estimated number of errors no detector found with its interval, and the
audit-cost reading. On NO-GO, name which condition failed.

In [ ]:
# GO / NO-GO, exactly PREREG 8. Nothing here is tunable.
truth_r = 181
est_r, union_r = PRIMARY_REDUX.n_hat, data_redux.n_found
mean_abs_est = float(np.mean([abs(r["n_hat"] - r["truth"]) for r in results]))
mean_abs_union = float(np.mean([abs(r["union"] - r["truth"]) for r in results]))
cond = {
    "a": coverage >= 0.85,
    "b": PRIMARY_REDUX.covers(truth_r),
    "c": (truth_r - union_r) / truth_r >= 0.20,
    "d": abs(est_r - truth_r) <= 0.5 * abs(union_r - truth_r) and mean_abs_est <= 0.5 * mean_abs_union,
}
detail = {
    "a": f"coverage {coverage:.1%} over {len(results)} replicates (need >= 85%)",
    "b": f"Redux CI [{PRIMARY_REDUX.lo:.1f}, {PRIMARY_REDUX.hi:.1f}] vs 181",
    "c": f"union finds {union_r}/181, misses {(truth_r - union_r) / truth_r:.1%} (need >= 20%)",
    "d": f"Redux |est-181| {abs(est_r - truth_r):.1f} vs half |union-181| {0.5 * abs(union_r - truth_r):.1f}; "
         f"replicates mean |est-266| {mean_abs_est:.1f} vs half mean |union-266| {0.5 * mean_abs_union:.1f}",
}
for k in "abcd":
    print(f"({k}) {'PASS' if cond[k] else 'FAIL'}  {detail[k]}")
if union_r >= 0.9 * truth_r or not cond["a"] or (not cond["b"] and not cond["d"]):
    verdict = "NO-GO"
elif all(cond.values()):
    verdict = "GO"
else:
    verdict = "PARTIAL (failed: " + ", ".join(k for k in "abcd" if not cond[k]) + ")"
print("\nreplicates: mean union", np.mean([r["union"] for r in results]),
      "| unstable share", np.mean([not r["ok"] for r in results]),
      "| median gof p", np.nanmedian([r["gof_p"] for r in results]))
print(f"\nPREREG.md blob {PREREG_COMMIT}")
print(f"VERDICT: {verdict}")
with open(os.path.join(DRIVE, "analysis_summary.json"), "w") as fh:
    json.dump({"verdict": verdict, "conditions": {k: bool(v) for k, v in cond.items()}, "detail": detail,
               "prereg_blob": PREREG_COMMIT,
               "redux": [{"name": e.name, "n_hat": e.n_hat, "lo": e.lo, "hi": e.hi, "ok": e.ok, "note": e.note}
                         for e in REDUX_EST],
               "replicates": [{k: (v if not isinstance(v, (list, tuple)) else str(v)) for k, v in r.items()}
                              for r in results]}, fh, indent=1, default=float)
print("saved analysis_summary.json to Drive")